In [1]:
import sys
import torch
import pickle
import os
from tqdm.notebook import tqdm

sys.path.insert(0, '..')
sys.path.insert(0, '../../')
sys.path.insert(0, '../../../')
sys.path.insert(0, '../../../../')
sys.path.insert(0, '../../../../../../')

from joinLSTM.model import FullShared_Join_LSTM
from robustness.camargo_evaluation import evaluate_seq_processing

os.environ["OMP_NUM_THREADS"] = "1"
os.environ["MKL_NUM_THREADS"] = "1"
os.environ["TORCH_NUM_THREADS"] = "1" 


In [ ]:
# Load model
file_path_model = '../../notebooks/training/BPIC17/BPIC17_camargo.pkl'
output_dir = '../../../../../evaluation_results/camargo/BPIC17/redo_activity/'
model = FullShared_Join_LSTM.load(file_path_model)

# Load datasets
file_path_original = '../../../../../encoded_data/BPIC17/BPIC_2017_all_5_test.pkl'
file_path_perturbed = '../../../../../encoded_data/BPIC17/BPIC_2017_all_5_test.pkl'
file_path_redo_activity = '../../../../../encoded_data/BPIC17/val.pkl'
file_path_redo_activity_pert = '../../../../../encoded_data/BPIC17/redo_activity.pkl'


original_dataset = torch.load(file_path_original, weights_only=False)
perturbed_dataset = torch.load(file_path_perturbed, weights_only=False)

redo_activity_dataset = torch.load(file_path_redo_activity, weights_only=False)
redo_activity_pert_dataset = torch.load(file_path_redo_activity_pert, weights_only=False)


print(f"Original dataset loaded: {len(original_dataset)} cases")
print(f"Perturbed dataset loaded: {len(perturbed_dataset)} cases")

Data set categories:  ([('concept:name', 28, {'A_Accepted': 1, 'A_Cancelled': 2, 'A_Complete': 3, 'A_Concept': 4, 'A_Create Application': 5, 'A_Denied': 6, 'A_Incomplete': 7, 'A_Pending': 8, 'A_Submitted': 9, 'A_Validating': 10, 'EOS': 11, 'O_Accepted': 12, 'O_Cancelled': 13, 'O_Create Offer': 14, 'O_Created': 15, 'O_Refused': 16, 'O_Returned': 17, 'O_Sent (mail and online)': 18, 'O_Sent (online only)': 19, 'W_Assess potential fraud': 20, 'W_Call after offers': 21, 'W_Call incomplete files': 22, 'W_Complete application': 23, 'W_Handle leads': 24, 'W_Personal Loan collection': 25, 'W_Shortened completion ': 26, 'W_Validate application': 27}), ('Action', 7, {'Created': 1, 'Deleted': 2, 'EOS': 3, 'Obtained': 4, 'Released': 5, 'statechange': 6}), ('org:resource', 150, {'EOS': 1, 'User_1': 2, 'User_10': 3, 'User_100': 4, 'User_101': 5, 'User_102': 6, 'User_103': 7, 'User_104': 8, 'User_105': 9, 'User_106': 10, 'User_107': 11, 'User_108': 12, 'User_109': 13, 'User_11': 14, 'User_110': 15, 'U

/home/chair/henryks_students/leon_urny/Robustness-in-suffix-prediction/.venv/lib64/python3.13/site-packages/torch/nn/modules/rnn.py:123: UserWarning: dropout option adds dropout after all but last recurrent layer, so non-zero dropout expects num_layers greater than 1, but got dropout=0.2 and num_layers=1
  warnings.warn(


Original dataset loaded: 248294 cases
Perturbed dataset loaded: 248294 cases


In [3]:
# Sample 10% of observations for faster inference
import random

# Set random seed for reproducibility (optional)
random.seed(17)

# Get all keys from both datasets
all_keys_orig = list(redo_activity_dataset.keys())
all_keys_pert = list(redo_activity_pert_dataset.keys())

# Ensure both datasets have the same keys
assert set(all_keys_orig) == set(all_keys_pert), "Datasets must have matching keys"

# Calculate 5% sample size
sample_size = max(1, int(len(all_keys_orig) * 0.05))

# Randomly sample 10% of the keys
sampled_keys = random.sample(all_keys_orig, sample_size)

# Create new dictionaries with only sampled keys
redo_activity_dataset = {key: redo_activity_dataset[key] for key in sampled_keys}
redo_activity_pert_dataset = {key: redo_activity_pert_dataset[key] for key in sampled_keys}

print(f"Sampled {len(sampled_keys)} observations ({len(sampled_keys)/len(all_keys_orig)*100:.1f}%) from {len(all_keys_orig)} total observations")
print(f"Original dataset now has {len(redo_activity_dataset)} entries")
print(f"Perturbed dataset now has {len(redo_activity_pert_dataset)} entries")

Sampled 8580 observations (5.0%) from 171611 total observations
Original dataset now has 8580 entries
Perturbed dataset now has 8580 entries


In [4]:
def save_chunk(results, i, output_dir):
    """Helper function to save intermediate results."""
    chunk_number = (i + 1)
    filename = os.path.join(output_dir, f'robustness_results_part_{chunk_number:03d}.pkl')
    with open(filename, 'wb') as f:
        pickle.dump(results, f)
    print(f"Saved {len(results)} results to {filename}")

In [5]:
#create models


from robustness.camargo_evaluation import evaluate_with_predefined_prefixes


os.makedirs(output_dir, exist_ok=True)

save_every = 50
results = {}

# Create evaluation generators
# eval_original = evaluate_seq_processing(
#     model=model,
#     dataset=original_dataset,
#     device=torch.device("cpu"),
#     samples_per_case=20,
#     random_order=False
# )

# eval_perturbed = evaluate_seq_processing(
#     model=model,
#     dataset=original_dataset,
#     device=torch.device("cpu"),
#     samples_per_case=20,
#     random_order=False
# )
evaluate_with_predefined_prefixes_normal = evaluate_with_predefined_prefixes(
    model=model,
    dataset=original_dataset,  # Still needed for encoder_decoder and categories
    predefined_pairs=redo_activity_dataset,
    device=torch.device("cpu"),
    samples_per_case=100,
    random_order=False,
    concept_name='concept:name',
)

evaluate_with_predefined_prefixes_pert = evaluate_with_predefined_prefixes(
    model=model,
    dataset=original_dataset,  # Still needed for encoder_decoder and categories
    predefined_pairs=redo_activity_pert_dataset,
    device=torch.device("cpu"),
    samples_per_case=100,
    random_order=False,
    concept_name='concept:name',
)


print("Evaluation generators created")

Evaluation generators created


In [6]:
# Main evaluation loop

for i, ((case_name_orig, prefix_len_orig, prefix_orig, sampled_cets_orig, suffix_orig, mean_cet_orig),
        (case_name_pert, prefix_len_pert, prefix_pert, sampled_cets_pert, suffix_pert, mean_cet_pert)) in enumerate(
        tqdm(zip(evaluate_with_predefined_prefixes_normal, evaluate_with_predefined_prefixes_pert), 
        desc="Evaluating robustness")):

    
    #Store results
    key = (case_name_orig, prefix_len_orig)
    results[key] = {
        'original': (prefix_orig, suffix_orig, mean_cet_orig, sampled_cets_orig),
        'perturbed': (prefix_pert, suffix_pert, mean_cet_pert, sampled_cets_pert)
    }
    
    if (i + 1) % save_every == 0:
        save_chunk(results, i, output_dir)
        results = {}

if len(results):
    save_chunk(results, i, output_dir)

print("Robustness evaluation completed!")

Evaluating robustness: 0it [00:00, ?it/s]

  0%|          | 0/8580 [00:00<?, ?it/s]

  0%|          | 0/8580 [00:00<?, ?it/s]

Saved 50 results to ../../../../../evaluation_results/camargo/BPIC17/random_event_attack_all/robustness_results_part_050.pkl


Saved 50 results to ../../../../../evaluation_results/camargo/BPIC17/random_event_attack_all/robustness_results_part_100.pkl


...

Saved 50 results to ../../../../../evaluation_results/camargo/BPIC17/random_event_attack_all/robustness_results_part_8400.pkl


Saved 50 results to ../../../../../evaluation_results/camargo/BPIC17/random_event_attack_all/robustness_results_part_8450.pkl


Saved 50 results to ../../../../../evaluation_results/camargo/BPIC17/random_event_attack_all/robustness_results_part_8500.pkl


Saved 50 results to ../../../../../evaluation_results/camargo/BPIC17/random_event_attack_all/robustness_results_part_8550.pkl


Saved 30 results to ../../../../../evaluation_results/camargo/BPIC17/random_event_attack_all/robustness_results_part_8580.pkl
Robustness evaluation completed!


In [7]:
# Load all saved chunks and combine them
all_results = {}
# Get all chunk files and sort them
chunk_files = [f for f in os.listdir(output_dir) if f.startswith('robustness_results_part_')]
chunk_files.sort()  # Ensure correct order

print(f"Found {len(chunk_files)} chunk files")

for chunk_file in chunk_files:
    chunk_path = os.path.join(output_dir, chunk_file)
    print(f"Loading {chunk_file}...")
    with open(chunk_path, 'rb') as f:
        chunk_results = pickle.load(f)
        all_results.update(chunk_results)
        print(f"  Added {len(chunk_results)} results from {chunk_file}")

# Also add the final results if any (e.g. from a still-running evaluation loop)
if 'results' in locals() and len(results) > 0:
    print(f"Adding final {len(results)} results...")
    all_results.update(results)

print(f"\nTotal results loaded: {len(all_results)}")

# Save combined results into a single pickle file
combined_results_path = os.path.join(output_dir, 'robustness_results.pkl')
with open(combined_results_path, 'wb') as f:
    pickle.dump(all_results, f)

print(f"Combined results saved to {combined_results_path}")



Found 172 chunk files
Loading robustness_results_part_050.pkl...
  Added 50 results from robustness_results_part_050.pkl
Loading robustness_results_part_100.pkl...
  Added 50 results from robustness_results_part_100.pkl
Loading robustness_results_part_1000.pkl...
  Added 50 results from robustness_results_part_1000.pkl
Loading robustness_results_part_1050.pkl...
  Added 50 results from robustness_results_part_1050.pkl
Loading robustness_results_part_1100.pkl...
  Added 50 results from robustness_results_part_1100.pkl
Loading robustness_results_part_1150.pkl...
  Added 50 results from robustness_results_part_1150.pkl
Loading robustness_results_part_1200.pkl...


  Added 50 results from robustness_results_part_1200.pkl
Loading robustness_results_part_1250.pkl...
  Added 50 results from robustness_results_part_1250.pkl
Loading robustness_results_part_1300.pkl...
  Added 50 results from robustness_results_part_1300.pkl
Loading robustness_results_part_1350.pkl...
  Added 50 results from robustness_results_part_1350.pkl
Loading robustness_results_part_1400.pkl...
  Added 50 results from robustness_results_part_1400.pkl
Loading robustness_results_part_1450.pkl...
  Added 50 results from robustness_results_part_1450.pkl
Loading robustness_results_part_150.pkl...
  Added 50 results from robustness_results_part_150.pkl
Loading robustness_results_part_1500.pkl...
  Added 50 results from robustness_results_part_1500.pkl
Loading robustness_results_part_1550.pkl...


  Added 50 results from robustness_results_part_1550.pkl
Loading robustness_results_part_1600.pkl...
  Added 50 results from robustness_results_part_1600.pkl
Loading robustness_results_part_1650.pkl...
  Added 50 results from robustness_results_part_1650.pkl
Loading robustness_results_part_1700.pkl...
  Added 50 results from robustness_results_part_1700.pkl
Loading robustness_results_part_1750.pkl...
  Added 50 results from robustness_results_part_1750.pkl
Loading robustness_results_part_1800.pkl...
  Added 50 results from robustness_results_part_1800.pkl
Loading robustness_results_part_1850.pkl...
  Added 50 results from robustness_results_part_1850.pkl
Loading robustness_results_part_1900.pkl...
  Added 50 results from robustness_results_part_1900.pkl
Loading robustness_results_part_1950.pkl...


  Added 50 results from robustness_results_part_1950.pkl
Loading robustness_results_part_200.pkl...
  Added 50 results from robustness_results_part_200.pkl
Loading robustness_results_part_2000.pkl...
  Added 50 results from robustness_results_part_2000.pkl
Loading robustness_results_part_2050.pkl...
  Added 50 results from robustness_results_part_2050.pkl
Loading robustness_results_part_2100.pkl...
  Added 50 results from robustness_results_part_2100.pkl
Loading robustness_results_part_2150.pkl...
  Added 50 results from robustness_results_part_2150.pkl
Loading robustness_results_part_2200.pkl...
  Added 50 results from robustness_results_part_2200.pkl
Loading robustness_results_part_2250.pkl...


  Added 50 results from robustness_results_part_2250.pkl
Loading robustness_results_part_2300.pkl...


...

  Added 50 results from robustness_results_part_7750.pkl
Loading robustness_results_part_7800.pkl...
  Added 50 results from robustness_results_part_7800.pkl
Loading robustness_results_part_7850.pkl...
  Added 50 results from robustness_results_part_7850.pkl
Loading robustness_results_part_7900.pkl...
  Added 50 results from robustness_results_part_7900.pkl
Loading robustness_results_part_7950.pkl...
  Added 50 results from robustness_results_part_7950.pkl
Loading robustness_results_part_800.pkl...
  Added 50 results from robustness_results_part_800.pkl


Loading robustness_results_part_8000.pkl...
  Added 50 results from robustness_results_part_8000.pkl
Loading robustness_results_part_8050.pkl...
  Added 50 results from robustness_results_part_8050.pkl
Loading robustness_results_part_8100.pkl...
  Added 50 results from robustness_results_part_8100.pkl
Loading robustness_results_part_8150.pkl...
  Added 50 results from robustness_results_part_8150.pkl
Loading robustness_results_part_8200.pkl...
  Added 50 results from robustness_results_part_8200.pkl
Loading robustness_results_part_8250.pkl...
  Added 50 results from robustness_results_part_8250.pkl
Loading robustness_results_part_8300.pkl...


  Added 50 results from robustness_results_part_8300.pkl
Loading robustness_results_part_8350.pkl...
  Added 50 results from robustness_results_part_8350.pkl
Loading robustness_results_part_8400.pkl...
  Added 50 results from robustness_results_part_8400.pkl
Loading robustness_results_part_8450.pkl...
  Added 50 results from robustness_results_part_8450.pkl
Loading robustness_results_part_850.pkl...
  Added 50 results from robustness_results_part_850.pkl
Loading robustness_results_part_8500.pkl...


  Added 50 results from robustness_results_part_8500.pkl
Loading robustness_results_part_8550.pkl...
  Added 50 results from robustness_results_part_8550.pkl
Loading robustness_results_part_8580.pkl...
  Added 30 results from robustness_results_part_8580.pkl
Loading robustness_results_part_900.pkl...
  Added 50 results from robustness_results_part_900.pkl
Loading robustness_results_part_950.pkl...
  Added 50 results from robustness_results_part_950.pkl
Adding final 30 results...

Total results loaded: 8580


Combined results saved to ../../../../../evaluation_results/camargo/BPIC17/random_event_attack_all/robustness_results.pkl
